In [1]:
import torch

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

PyTorch Version: 2.11.0+cu128
GPU Available: True
GPU Name: Tesla T4


In [2]:
!pip -q install "transformers>=4.45,<5.0" datasets evaluate accelerate gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.5 MB/s eta 0:00:00


In [3]:
import transformers
import datasets
import evaluate
import accelerate
import gradio

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)
print("Accelerate:", accelerate.__version__)
print("Gradio:", gradio.__version__)

Transformers: 4.57.6
Datasets: 4.0.0
Evaluate: 0.4.6
Accelerate: 1.14.0
Gradio: 6.17.3


In [4]:
from datasets import load_dataset

# Load AG News dataset
dataset = load_dataset("fancyzhx/ag_news")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [5]:
# Print dataset information

print("Training Samples:", len(dataset["train"]))
print("Testing Samples:", len(dataset["test"]))

print("\nFirst Training Example:\n")
print(dataset["train"][0])

print("\nLabel Mapping:")
labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

for key, value in labels.items():
    print(f"{key} -> {value}")

Training Samples: 120000
Testing Samples: 7600

First Training Example:

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}

Label Mapping:
0 -> World
1 -> Sports
2 -> Business
3 -> Sci/Tech


In [6]:
# Create a smaller subset for faster training

train_dataset = dataset["train"].shuffle(seed=42).select(range(10000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(2000))

print("Training Samples:", len(train_dataset))
print("Testing Samples:", len(test_dataset))

Training Samples: 10000
Testing Samples: 2000


In [7]:
from transformers import AutoTokenizer

# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!


In [8]:
# Tokenize one example

sample_text = train_dataset[0]["text"]

encoded = tokenizer(
    sample_text,
    truncation=True,
    padding="max_length",
    max_length=128
)

print("Original Text:\n")
print(sample_text)

print("\nInput IDs:\n")
print(encoded["input_ids"])

print("\nAttention Mask:\n")
print(encoded["attention_mask"])

Original Text:

Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.

Input IDs:

[101, 7269, 11498, 2135, 6924, 2011, 9326, 4559, 10134, 2031, 2716, 2116, 4865, 1998, 3655, 1999, 7269, 2000, 1037, 9190, 1010, 1996, 2154, 2044, 2324, 2111, 2351, 1999, 18217, 2012, 1037, 2576, 8320, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Attention Mask:

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [9]:
# Function to tokenize the dataset
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Tokenization completed successfully!")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenization completed successfully!


In [10]:
from transformers import AutoModelForSequenceClassification

# Load pre-trained BERT model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

print("BERT model loaded successfully!")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT model loaded successfully!


In [11]:
import numpy as np
import evaluate

# Load evaluation metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

# Function to compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=predictions, references=labels)

    f1_score = f1.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

print("Evaluation metrics ready!")

Evaluation metrics ready!


In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_news_classifier",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    report_to="none"
)

print("Training arguments configured successfully!")

Training arguments configured successfully!


In [13]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer created successfully!")

Trainer created successfully!


In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.276400,0.275738,0.906500,0.906243
2,0.171400,0.277165,0.916000,0.915908


TrainOutput(global_step=1250, training_loss=0.27979118041992185, metrics={'train_runtime': 517.4254, 'train_samples_per_second': 38.653, 'train_steps_per_second': 2.416, 'total_flos': 1315578900480000.0, 'train_loss': 0.27979118041992185, 'epoch': 2.0})

In [15]:
# Evaluate the trained model

results = trainer.evaluate()

print("\nFinal Evaluation Results:")
print(results)


Final Evaluation Results:
{'eval_loss': 0.2757381498813629, 'eval_accuracy': 0.9065, 'eval_f1': 0.9062430889537598, 'eval_runtime': 20.3028, 'eval_samples_per_second': 98.508, 'eval_steps_per_second': 6.157, 'epoch': 2.0}


In [16]:
# Save the trained model and tokenizer

model.save_pretrained("bert_news_classifier")

tokenizer.save_pretrained("bert_news_classifier")

print("Model saved successfully!")

Model saved successfully!


In [17]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load saved model
tokenizer = AutoTokenizer.from_pretrained("bert_news_classifier")
model = AutoModelForSequenceClassification.from_pretrained("bert_news_classifier")

# Labels
labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

# Prediction function
def predict_news(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    return labels[prediction]

# Create Gradio Interface
demo = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Enter a news headline or article..."
    ),
    outputs="text",
    title="News Topic Classifier using BERT",
    description="Predicts whether a news article belongs to World, Sports, Business, or Sci/Tech."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://77f2d3403be04c6f6c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
